# Example 02: creat a simple robot solver manager

In this code you will learn how to create a robot configuration file and generate a robot solver manager `ManagerMpcAcados` for the robot base hunter.

We import the `ModelCfg` defined in example 01. In this example we will focus on the configuration of the cost function and constraints of the MPC.

In [ ]:
import numpy as np
import casadi as ca
from casadi import (
    vertcat,
    horzcat,
    cos,
    sin,
    exp,
    norm_2,
    sqrt
)
from o2r_pi2_controllers.utils import configclass, ca_euler2rot, ca_rot2euler
from o2r_pi2_controllers.managers.manager_mpc_acados import ManagerMpcAcados, CostTerm, HardConstTerm, SoftConstTerm
from o2r_pi2_controllers.robots_cfg.hunter import ModelCfg

___
## Constraints Cfg

Let's start with the constraint configuration. It requires the attributes `lbu`, `ubu` `lbx` and `ubx`, witch are acados requirement for lower and upper bands for state and command. 

In [ ]:
@configclass
class ConstCfg:
    lbu = -np.array([0.5,0.5])
    ubu =  np.array([0.5,0.5])

    lbx = -np.array([1000.,1000., 1000., 0.7])
    ubx =  np.array([1000.,1000., 1000., 0.7])


Constraint configuration also require `const_terms` witch is a list of constraints.

We first defined a constraint function as a ConstCfg method, taking as argument u, x, obstacles, target, and returning a list.

NOTE: For constraints and cost function you will find different types of arguments: u and x relative to the command and the state of the robot, and traj_base, traj_arm, target, obstacles. Those last are parameters given to the solver, there are initiate with a certain length (7,7,3,0) but those values can be modified via traj_base_len, traj_arm_len, target_len and obstacles_len. After modifying those values, you need to run `update_ref_len()`. 
Warning: you cannot modify parameters length after had creating the solver.


Here `collision_point_A` is a constraint on the distance from the robot to the point A.

In [ ]:
    def collision_point_A(self, u, x, obstacles, target):
        base = self.T_base(x)
        A = [2,5,0]
        const_point_A = sqrt((base[:3,3][0]-A[0])**2 + (base[:3,3][1]-A[1])**2 + (base[:3,3][2]-A[2])**2)
        return [const_point_A]

Then we create `const_terms` as a list of `HardConstTerm` and `SoftConstTerm`: a custom tuple regrouping a function and limits lh, uh, lh_e and uh_e (lower and upper limits of the constraint).

For each terms of the function output list, a constraint will be created with the limits lh, uh, lh_e and lu_e. 

Here we set a hard and soft constraints to stay away of point A from 0.5m.

In [ ]:
    const_terms = [
        HardConstTerm(func=collision_point_A, lh=0.5, uh=10e3, lh_e=0.5, uh_e=10e3),
        SoftConstTerm(func=collision_point_A, lh=1., uh=10e3, lh_e=1., uh_e=10e3),
    ]

At the end the constraint configuration class look like this:

In [ ]:
@configclass
class ConstCfg:
    lbu = -np.array([0.5,0.5])
    ubu =  np.array([0.5,0.5])

    lbx = -np.array([1000.,1000., 1000., 0.7])
    ubx =  np.array([1000.,1000., 1000., 0.7])

    def collision_point_A(self, u, x, obstacles, target):
        base = self.T_base(x)
        A = [2,5,0]
        const_point_A = sqrt((base[:3,3][0]-A[0])**2 + (base[:3,3][1]-A[1])**2 + (base[:3,3][2]-A[2])**2)
        return [const_point_A]
    
    const_terms = [
        HardConstTerm(func=collision_point_A, lh=0.5, uh=10e3, lh_e=0.5, uh_e=10e3),
        SoftConstTerm(func=collision_point_A, lh=1., uh=10e3, lh_e=1., uh_e=10e3),
    ]

___
## Cost function Cfg

We then defined the cost configuration. It requires the attributes `coefs_keys`, witch is a dictionary regrouping the weights name and dimension.

In these examples we will define two costs relative to the trajectory following and velocity.

In [ ]:
@configclass
class CostCfg:
    coefs_keys = {'wPos_base':1, 'qVelBase':2}


Cost configuration also require `cost_terms` witch is a list of cost function terms.

We first defined a cost function term as a CostCfg method, taking as argument u, x, traj_base, traj_arm, target, obstacles:

In [ ]:
    def cost_base_traj(self, u, x, traj_base, traj_arm, target, obstacles):
        T_front_base = self.T_base_front(x)
        e_base_traj = T_front_base[:2, 3] - traj_base[:2]
        a = 1.
        e = 1 - exp(-a*norm_2(e_base_traj)**2)
        return e

    def cost_vel_base(self, u, x, traj_base, traj_arm, target, obstacles):
        return u / self.ubu

Note: You can see that we have access to `ubu` via self, an attribute defined in the constraint configuration. This is possible because both configurations will be give to the solver manager at the end.

Then we create `cost_terms` as a list of `CostTerm`: a custom tuple regrouping a function and the relative weight name:

In [ ]:
    cost_terms = [
        CostTerm(func=cost_base_traj, weight_key='wPos_base'),
        CostTerm(func=cost_vel_base, weight_key='qVelBase'),
    ]

At the end the cost function configuration class look like this:

In [ ]:
@configclass
class CostCfg:
    coefs_keys = {'wPos_base':1, 'qVelBase':2}

    def cost_base_traj(self, u, x, traj_base, traj_arm, target, obstacles):
        T_front_base = self.T_base_front(x)
        e_base_traj = T_front_base[:2, 3] - traj_base[:2]
        a = 1.
        e = 1 - exp(-a*norm_2(e_base_traj)**2)
        return e

    def cost_vel_base(self, u, x, traj_base, traj_arm, target, obstacles):
        return u / self.ubu
    
    cost_terms = [
        CostTerm(func=cost_base_traj, weight_key='wPos_base'),
        CostTerm(func=cost_vel_base, weight_key='qVelBase'),
    ]

___
## ManagerMpcAcados: solver manager

Now that the model configuration is ready, create the robot configuration. In future examples it will be possible to add it other types of configurations, as ros msgs.

In [ ]:
@configclass
class HunterCfg():
    robot_model = ModelCfg()
    cost = CostCfg()
    const = ConstCfg()

Now that the robot configuration is ready, you can create the solver manager:

In [ ]:
hunter = ManagerMpcAcados(HunterCfg())

This solver manager is build with ManagerCasadiModel, so it as the same methods as expressed in example 01.

More than the models methods, this new class allow you to integer state and solve MPC problems.


In [ ]:
print("robot model:", hunter.name)
x = hunter.init_state()    
u = np.zeros(hunter.nu)
dt = 0.01
horizon = 20

# get in toutch with parameters
nb_obstacles = 0
hunter.obstacles_len = 7*nb_obstacles # In this software obstacles are set as capsules [pos A, pos B, radius]
hunter.update_ref_len() # update the length of the parameters vector after obstacles length is set
cost_coefs = {'wPos_base': {'_1': 1.}, 'qVelBase': {'_1': 1., '_2': 1.}}
cost_coefs_refs = hunter.extract_cost_coefs(cost_coefs)
print("Cost: ",hunter.compute_cost(u, x,
                        cost_coefs_refs, 
                        np.zeros((hunter.traj_base_len)), 
                        np.zeros((hunter.traj_arm_len)), 
                        np.zeros((hunter.target_len)), 
                        np.zeros((hunter.obstacles_len))))


# Integrator
hunter.create_integrator(dt, True)

hunter.integrator.set("x", x)
hunter.integrator.set("u", u)
status = hunter.integrator.solve()
if status != 0:
    print(f"integrator error: status {status} =! 0")

x = hunter.integrator.get("x")
print("integrated state:", x)

# MPC Solver
time_steps = np.array([dt for i in range(20)])  # time_steps
hunter.create_ocp_solver(x, horizon,
                            time_steps,
                            sum(time_steps), # tf
                            True, True)

refs = [] # set the references for every step of the MPC solver
for j in range(horizon+1): # +1 for the terminal state
    refs.append( np.hstack((
        cost_coefs_refs, 
        np.zeros((hunter.traj_base_len)), 
        np.zeros((hunter.traj_arm_len)), 
        np.zeros((hunter.target_len)), 
        np.zeros((hunter.obstacles_len))
    ))) # references should always be in this order: cost_coefs, traj_base, traj_arm, target, obstacles
print(refs)
x_mpc, u_mpc, status = hunter.solve_ocp(x, horizon, refs)
print("MPC commands:", u_mpc) # list of commands for each step of the MPC solver
print("MPC status:", status)